# Q5: Pattern Analysis

**Phase 6:** Pattern Analysis & Advanced Visualization  
**Points: 6 points**

**Focus:** Identify trends over time, analyze seasonal patterns, create
correlation analysis.

**Lecture Reference:** See **Lecture 11, Notebook 3**
(`11/demo/03_pattern_analysis_modeling_prep.ipynb`), Phase 6 for
examples of trend analysis, seasonal pattern identification, and
advanced visualizations. Also see **Lecture 09** for time series pattern
analysis.

## Setup

In [89]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Load feature-engineered data from Q4
df = pd.read_csv('output/q4_features.csv', parse_dates=['Measurement Timestamp'], index_col='Measurement Timestamp')
# Or if you saved without index:
# df = pd.read_csv('output/q4_features.csv')
# df['Measurement Timestamp'] = pd.to_datetime(df['Measurement Timestamp'])
# df = df.set_index('Measurement Timestamp')
print(f"Loaded {len(df):,} records with features")

# inspect column names before doing any work
print(df.columns)

Loaded 195,892 records with features
Index(['Station Name', 'Air Temperature', 'Wet Bulb Temperature', 'Humidity',
       'Rain Intensity', 'Interval Rain', 'Total Rain', 'Precipitation Type',
       'Wind Direction', 'Wind Speed', 'Maximum Wind Speed',
       'Barometric Pressure', 'Solar Radiation', 'Heading', 'Battery Life',
       'Measurement Timestamp Label', 'Measurement ID', 'hour', 'day_of_week',
       'month', 'year', 'day_name', 'is_weekend', 'day_of_month', 'quarter',
       'Temperature Difference', 'Temp Ratio', 'Wind Speed Squared',
       'Air Temperature (F)', 'Comfort Index', 'Air Temperature Categories',
       'Wind Speed Categories', 'pressure_rolling_7h',
       'barometric_pressure_rolling_mean_7h',
       'solar_radiation_rolling_mean_7h', 'total_rain_rolling_mean_24h'],
      dtype='object')


## Correlations

In [90]:
# SAVE ARTIFACT 1: q5_correlations.csv
# Get all numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nTotal numeric columns: {len(numeric_cols)}")

# Identify key sensor variables (exclude derived/temporal features for clarity)
# You may want to focus on original sensor readings
temporal_features = ['hour', 'day_of_week', 'month', 'year', 'is_weekend', 
                    'day_of_month', 'quarter']

important_vars = [
    'Air Temperature',
    'Wet Bulb Temperature', 
    'Humidity',
    'Rain Intensity',
    'Total Rain',
    'Wind Speed',
    'Barometric Pressure',
    'Solar Radiation',
    # New features from Q4
    "Temperature Difference",
    "Wind Speed Squared",
    "Comfort Index",
    "Temp Ratio",
    # Rolling features
    "pressure_rolling_7h",
    "barometric_pressure_rolling_mean_7h",
    "solar_radiation_rolling_mean_7h",
    "total_rain_rolling_mean_24h",
]
# Select numeric columns for correlation (exclude categorical)
corr_cols = [col for col in important_vars if df[col].dtype in [np.float64, np.int64]]

# Calculate correlation matrix
corr_matrix = df[corr_cols].corr()

# Find strongest correlations (excluding diagonal)
print("\nStrongest positive correlations (top 5):")
# Get upper triangle indices
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
corr_unstacked = corr_matrix.where(mask).stack().sort_values(ascending=False)
print(corr_unstacked.head(5))

print("\nStrongest negative correlations (top 5):")
print(corr_unstacked.tail(5))

# Store key correlations for summary
top_positive = corr_unstacked.head(3)
top_negative = corr_unstacked.tail(3)

# Save correlation matrix to CSV
corr_matrix.to_csv('output/q5_correlations.csv')
print("✓ Saved output/q5_correlations.csv")


Total numeric columns: 30

Strongest positive correlations (top 5):
Air Temperature       Comfort Index                          0.999965
pressure_rolling_7h   barometric_pressure_rolling_mean_7h    0.992115
Barometric Pressure   pressure_rolling_7h                    0.991215
Wet Bulb Temperature  Comfort Index                          0.979465
Air Temperature       Wet Bulb Temperature                   0.978050
dtype: float64

Strongest negative correlations (top 5):
Wet Bulb Temperature  barometric_pressure_rolling_mean_7h   -0.253040
                      pressure_rolling_7h                   -0.263505
                      Barometric Pressure                   -0.268354
Humidity              solar_radiation_rolling_mean_7h       -0.270066
                      Temperature Difference                -0.701920
dtype: float64
✓ Saved output/q5_correlations.csv


In [91]:
# TEMPORAL TREND ANALYSIS
# Monthly averages (use 'ME' for month end)
monthly_avg = df[important_vars].resample('ME').mean()
# Daily averages
daily_avg = df[important_vars].resample('D').mean()

# Calculate overall trends
print("\nOverall trend analysis:")
for var in important_vars[:5]:  # Analyze first 5 variables
    overall_mean = df[var].mean()
    overall_std = df[var].std()
    overall_min = df[var].min()
    overall_max = df[var].max()
    print(f"- {var}: Mean={overall_mean:.2f}, Std={overall_std:.2f}, Min={overall_min:.2f}, Max={overall_max:.2f}")

# SEASONAL PATTERN ANALYSIS
# Monthly patterns
if 'month' in df.columns:
    monthly_pattern = df.groupby('month')[important_vars].mean()

# Hourly patterns (diurnal cycle)
if 'hour' in df.columns:
    hourly_pattern = df.groupby('hour')[important_vars].mean()

# Day of week patterns
if 'day_of_week' in df.columns:
    dow_pattern = df.groupby('day_of_week')[important_vars].mean()

# Weekend vs weekday patterns
if 'is_weekend' in df.columns:
    weekend_pattern = df.groupby('is_weekend')[important_vars].mean()
    if len(important_vars) > 0:
        weekday_val = weekend_pattern.loc[0, important_vars[0]]
        weekend_val = weekend_pattern.loc[1, important_vars[0]]



Overall trend analysis:
- Air Temperature: Mean=12.65, Std=10.43, Min=-29.78, Max=37.60
- Wet Bulb Temperature: Mean=10.02, Std=9.40, Min=-28.90, Max=28.40
- Humidity: Mean=68.02, Std=15.64, Min=0.00, Max=100.00
- Rain Intensity: Mean=0.00, Std=0.00, Min=0.00, Max=0.00
- Total Rain: Mean=129.19, Std=168.30, Min=0.00, Max=699.70


In [92]:
# Create figure with 4 subplots
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)

# Choose primary variable for visualization (first key variable)
primary_var = important_vars[0] if len(important_vars) > 0 else numeric_cols[0]
secondary_var = important_vars[1] if len(important_vars) > 1 else numeric_cols[1]

# Plot 1: Monthly trend over time 
ax1 = fig.add_subplot(gs[0, :])
if len(important_vars) >= 2:
    ax1.plot(monthly_avg.index, monthly_avg[primary_var], 
            linewidth=2, color='coral', marker='o', label=primary_var)
    ax1_twin = ax1.twinx()
    ax1_twin.plot(monthly_avg.index, monthly_avg[secondary_var], 
                linewidth=2, color='steelblue', marker='s', label=secondary_var)
    ax1.set_xlabel('Date', fontsize=12)
    ax1.set_ylabel(primary_var, fontsize=12, color='coral')
    ax1_twin.set_ylabel(secondary_var, fontsize=12, color='steelblue')
    ax1.set_title('Temporal Trends: Monthly Averages', fontsize=14, fontweight='bold')
    ax1.grid(alpha=0.3)
    ax1.legend(loc='upper left')
    ax1_twin.legend(loc='upper right')
    ax1.tick_params(axis='y', labelcolor='coral')
    ax1_twin.tick_params(axis='y', labelcolor='steelblue')
else:
    ax1.plot(monthly_avg.index, monthly_avg[primary_var], 
            linewidth=2, color='coral', marker='o')
    ax1.set_xlabel('Date', fontsize=12)
    ax1.set_ylabel(primary_var, fontsize=12)
    ax1.set_title('Temporal Trend: Monthly Average', fontsize=14, fontweight='bold')
    ax1.grid(alpha=0.3)

plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45)

# Plot 2: Monthly seasonal pattern (middle-left)
ax2 = fig.add_subplot(gs[1, 0])
if 'month' in df.columns and len(important_vars) > 0:
    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    ax2.bar(range(1, 13), monthly_pattern[primary_var], 
            color='skyblue', edgecolor='black', alpha=0.7)
    ax2.set_xlabel('Month', fontsize=12)
    ax2.set_ylabel(primary_var, fontsize=12)
    ax2.set_title(f'Monthly Seasonal Pattern: {primary_var}', fontsize=12, fontweight='bold')
    ax2.set_xticks(range(1, 13))
    ax2.set_xticklabels(month_names, rotation=45)
    ax2.grid(axis='y', alpha=0.3)

# Plot 3: Hourly diurnal pattern (middle-right)
ax3 = fig.add_subplot(gs[1, 1])
if 'hour' in df.columns and len(important_vars) > 0:
    ax3.plot(range(24), hourly_pattern[primary_var], 
            linewidth=2.5, color='darkorange', marker='o', markersize=6)
    ax3.set_xlabel('Hour of Day', fontsize=12)
    ax3.set_ylabel(primary_var, fontsize=12)
    ax3.set_title(f'Diurnal Pattern: {primary_var}', fontsize=12, fontweight='bold')
    ax3.set_xticks(range(0, 24, 3))
    ax3.grid(alpha=0.3)
    # Shade night hours
    ax3.axvspan(0, 6, alpha=0.1, color='gray', label='Night')
    ax3.axvspan(18, 24, alpha=0.1, color='gray')
    ax3.legend()

# Plot 4: Correlation heatmap
ax4 = fig.add_subplot(gs[2, :])
# Select subset of variables for clearer heatmap
if len(corr_cols) > 10:
    # Show top 10 most correlated variables with primary variable
    if primary_var in corr_matrix.columns:
        top_corr_vars = corr_matrix[primary_var].abs().nlargest(11).index.tolist()
        heatmap_data = corr_matrix.loc[top_corr_vars, top_corr_vars]
    else:
        heatmap_data = corr_matrix.iloc[:10, :10]
else:
    heatmap_data = corr_matrix

sns.heatmap(heatmap_data, annot=True, annot_kws={"fontsize": 8}, fmt='.2f', cmap='coolwarm', 
            center=0, vmin=-1, vmax=1, square=True, 
            linewidths=0.5, linecolor='white', cbar_kws={"shrink": 0.8}, ax=ax4)
ax4.set_title('Correlation Matrix Heatmap', fontsize=14, fontweight='bold')
plt.setp(ax4.get_xticklabels(), rotation=45, ha='right', fontsize=10)
plt.setp(ax4.get_yticklabels(), rotation=0, fontsize=10)

# Overall title
fig.suptitle('Chicago Beach Weather Sensors - Pattern Analysis', 
            fontsize=16, fontweight='bold', y=0.995)

# Save figure
plt.savefig('output/q5_patterns.png', dpi=150, bbox_inches='tight')
print("✓ Saved: output/q5_patterns.png")
plt.close()


✓ Saved: output/q5_patterns.png


In [93]:
# Calculate key statistics for summary
if 'month' in df.columns and len(important_vars) > 0:
    monthly_range = (monthly_pattern[primary_var].min(), 
                    monthly_pattern[primary_var].max())
    peak_month = monthly_pattern[primary_var].idxmax()
    low_month = monthly_pattern[primary_var].idxmin()
else:
    monthly_range = (df[primary_var].min(), df[primary_var].max())
    peak_month = "N/A"
    low_month = "N/A"

if 'hour' in df.columns and len(important_vars) > 0:
    peak_hour = hourly_pattern[primary_var].idxmax()
    low_hour = hourly_pattern[primary_var].idxmin()
else:
    peak_hour = "N/A"
    low_hour = "N/A"

# Write summary
with open('output/q5_trend_summary.txt', 'w') as f:
    f.write("KEY PATTERNS IDENTIFIED\n")
    f.write("=" * 60 + "\n\n")
    
    f.write("TEMPORAL TRENDS:\n")
    f.write(f"- Primary variable analyzed: {primary_var}\n")
    f.write(f"- Overall mean: {df[primary_var].mean():.2f}\n")
    f.write(f"- Overall std: {df[primary_var].std():.2f}\n")
    f.write(f"- Overall range: [{df[primary_var].min():.2f}, {df[primary_var].max():.2f}]\n")
    f.write(f"- Monthly range: [{monthly_range[0]:.2f}, {monthly_range[1]:.2f}]\n")
    
    if peak_month != "N/A":
        month_names = {1: 'January', 2: 'February', 3: 'March', 4: 'April',
                    5: 'May', 6: 'June', 7: 'July', 8: 'August',
                    9: 'September', 10: 'October', 11: 'November', 12: 'December'}
        f.write(f"- Peak month: {month_names.get(peak_month, peak_month)}\n")
        f.write(f"- Lowest month: {month_names.get(low_month, low_month)}\n")
    
    f.write("\nDAILY PATTERNS:\n")
    if peak_hour != "N/A":
        f.write(f"- {primary_var} shows diurnal cycle\n")
        f.write(f"- Peak hour: {peak_hour}:00 ({hourly_pattern[primary_var].max():.2f})\n")
        f.write(f"- Minimum hour: {low_hour}:00 ({hourly_pattern[primary_var].min():.2f})\n")
        diurnal_range = hourly_pattern[primary_var].max() - hourly_pattern[primary_var].min()
        f.write(f"- Daily temperature range: {diurnal_range:.2f}\n")
    else:
        f.write("- Daily patterns not analyzed (hour feature not available)\n")
    
    f.write("\nCORRELATIONS:\n")
    f.write("Top positive correlations:\n")
    for idx, (pair, corr) in enumerate(top_positive.items(), 1):
        f.write(f"  {idx}. {pair[0]} vs {pair[1]}: {corr:.3f}\n")
    
    f.write("\nTop negative correlations:\n")
    for idx, (pair, corr) in enumerate(top_negative.items(), 1):
        f.write(f"  {idx}. {pair[0]} vs {pair[1]}: {corr:.3f}\n")
    
    f.write("\nKEY INSIGHTS:\n")
    if peak_month != "N/A" and peak_month in [6, 7, 8]:
        f.write(f"- {primary_var} peaks in summer months\n")
    if peak_month != "N/A" and peak_month in [12, 1, 2]:
        f.write(f"- {primary_var} lowest in winter months\n")
    if peak_hour != "N/A" and peak_hour in range(12, 17):
        f.write(f"- {primary_var} peaks in afternoon hours\n")

print("✓ Saved: output/q5_trend_summary.txt")


✓ Saved: output/q5_trend_summary.txt
